# colabel pipeline (merged)

Single notebook covering the full §2.2 label-completion pipeline, previously split across
three notebooks (`colabel_summary.ipynb`, `structure_DPA.ipynb` cell 3, `colabel.ipynb`):

1. high-similarity clustering (mmseqs2, 95% seq-id) -> `high_id_cluster/`
2. per-cluster summary table -> `all_cluster_summary.csv`
3. pairwise-align every cluster member to its reference (longest) sequence -> `colabel_mask_all.pkl`
4. co-label: OR-union each member's own label onto the reference frame, per cluster -> `all_colabel.pkl`
5. (appendix, optional) single-cluster visual sanity check + a preview of the pocket
   sub-clustering idea (Tanimoto/Butina) that was later developed into its own
   `colabel_pocket_clustering.ipynb` / cluster pocket-clustering pipeline.

Merged from the original three notebooks; step 2's CSV-writer body was replaced with the
leaner, structure-independent version from `structure_DPA.ipynb` (only needs
`high_id_cluster/all_cluster.txt` + `all_seq_dict.pkl`, not a live `Dataset`/h5 handle) so
this step can be re-run cheaply on its own. Step 5 is unchanged from `colabel.ipynb`'s only
non-visualization contribution (all_colabel.pkl is written in step 4 of the ORIGINAL
colabel_summary.ipynb, not in colabel.ipynb -- colabel.ipynb turned out to be pure
example/visualization on top of already-computed outputs, no unique pipeline artifact).

## 0. Setup: load Dataset + reference sequence dict

In [ ]:
import os
import pickle
import numpy as np
from data_handler import Dataset
from src.dataset import select_by_sid, select_by_interface_types
from config import config_data

import pandas as pd
import matplotlib.pyplot as plt
import ast
import numpy as np


dataset_path = "../bio_assemblies_251210.h5"

with open('../cluster/all_seq_dict.pkl', 'rb') as d:
    all_seq_dict = pickle.load(d)

dataset = Dataset(config_data)

## 1. Build FASTA of all dataset chains

In [ ]:
dataset_name = 'all'
dataset_dir = '../colabel'
sel_m = np.ones(len(dataset.keys), dtype=bool)

sel_m &= (dataset.sizes[:, 1] >= config_data['min_num_res'])
sel_m &= (dataset.sizes[:, 0] <= config_data['max_size'])
sel_m &= select_by_interface_types(dataset, config_data['l_types'])  # select by interface type

dataset.update_mask(sel_m)
print(len(dataset))
dataset.set_types(config_data['l_types'], config_data['r_types'])


parts = np.array([p for p in np.char.split(np.array(dataset.ukeys), "/")])
with open('all_id.txt', 'w') as a:
    for line in parts:
        a.write('/'.join(line) + '\n')
pdbid = parts[:, 1]
chainid = np.array([c for c in np.char.split(parts[:, 2], ":")])[:, 0]
train_id = np.array([f"{p.upper()}_{c}" for p, c in zip(pdbid, chainid.astype(str))])


if not os.path.exists('high_id_cluster'):
    os.mkdir('high_id_cluster')
with open(f"high_id_cluster/{dataset_name.split('.')[0]}.fasta", 'w') as w:
    for id in dataset.ukeys:
        pdbid = id.split('/')[1]
        chainid = id.split('/')[2].split(':')[0]
        if f"{pdbid}_{chainid}" in all_seq_dict:
            w.write(f">{id}\n{all_seq_dict[f'{pdbid}_{chainid}']}\n")
        elif f'{pdbid}_{chainid.lower()}' in all_seq_dict:
            w.write(f">{id}\n{all_seq_dict[f'{pdbid}_{chainid.lower()}']}\n")
        else:
            w.write(f">{id}\n{all_seq_dict[f'{pdbid}_{chainid.upper()}']}\n")

## 2. High-similarity clustering (mmseqs2, 95% seq-id)

In [ ]:
!mmseqs easy-cluster high_id_cluster/{dataset_name.split('.')[0]}.fasta high_id_cluster/{dataset_name.split('.')[0]} tmp --min-seq-id 0.95 -c 0.9 --cov-mode 0

Parse mmseqs' `_cluster.tsv` into a flat `cluster_id -> members` text file.

In [ ]:
def handle_tsv(input_file, output_file):
    cluster_name = []
    name_cluster = {}
    cur_list = []
    cluster_name_dict = {}
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        cur_c = ""
        for line in infile:
            words = line.strip().split()
            if words[0] != cur_c:
                if cur_list:
                    outfile.write((' ').join(cur_list) + '\n')
                    cur_list = []
                cur_c = words[0]
                cluster_name_dict[cur_c] = []
                cluster_name_dict[cur_c].append(words[1])
                name_cluster[words[1]] = cur_c
                cluster_name.append(cur_c)
                cur_list.append(words[1])
            else:
                cluster_name_dict[cur_c].append(words[1])
                cur_list.append(words[1])
                name_cluster[words[1]] = cur_c
        if cur_list:
            outfile.write((' ').join(cur_list) + '\n')
    return cluster_name_dict


input_file = f"high_id_cluster/{dataset_name.split('.')[0]}_cluster.tsv"
output_file = f"high_id_cluster/{dataset_name.split('.')[0]}_cluster.txt"
cluster_name_dict = handle_tsv(input_file, output_file)
print(('X3/2X3W/D:0' in cluster_name_dict))

## 3. Per-cluster summary table -> `all_cluster_summary.csv`

**Merged in from `structure_DPA.ipynb` cell 3** (that notebook's whole purpose was
regenerating this CSV standalone after the source tree's volatility bug wiped intermediate
files -- see `[[interface-prediction-dir-volatile]]`). Kept here because it only needs
`high_id_cluster/{name}_cluster.txt` (written above) + `all_seq_dict` (already in memory
from step 0), not a live `Dataset`/h5 handle -- so this cell alone is safe to re-run if the
h5-backed `dataset` object above ever needs reloading. Same 5 output columns as the
original in-notebook version (`cluster_names, cluster_size, seq_name, seq_length, min_max`).

In [ ]:
def ukey_to_ref_key(ukey):
    """ukey like I4/8I4S/A:0 -> all_seq_dict key 8I4S_A (matches model/utils convention)."""
    p = ukey.strip().split("/")
    return p[1].upper() + "_" + p[2].split(":")[0]


with open(output_file, "r") as t:
    raw_data = t.read()

summary_rows = []
for line in raw_data.split("\n"):
    if not line.strip():
        continue
    members = [c.strip() for c in line.split() if c.strip()]
    center = members[0]
    seq_length = []
    for u in members:
        k = ukey_to_ref_key(u)
        seq = all_seq_dict.get(k)
        seq_length.append(len(seq) if seq is not None else -1)
    pos_lens = [L for L in seq_length if L >= 0]
    min_max = (min(pos_lens), max(pos_lens)) if pos_lens else (-1, -1)
    summary_rows.append({
        "cluster_names": center,
        "cluster_size": len(members),
        "seq_name": str(members),
        "seq_length": str(seq_length),
        "min_max": str(min_max),
    })

df = pd.DataFrame(summary_rows)
df.to_csv(
    f"{dataset_name.split('.')[0]}_cluster_summary.csv",
    index=True,
    columns=["cluster_names", "cluster_size", "seq_name", "seq_length", "min_max"],
)
print(f"wrote {len(df)} rows to {dataset_name.split('.')[0]}_cluster_summary.csv")

Cluster-size distribution (unchanged from the original notebook).

In [ ]:
df = pd.read_csv(f"{dataset_name.split('.')[0]}_cluster_summary.csv")

bins = [0, 1, 5, 10, 100, 1000, 10000]
labels = ["1", "2-5", "5-10", "11-100", "101-1000", "1000-10000"]
df['size_bin'] = pd.cut(df['cluster_size'], bins=bins, labels=labels, right=True)

bin_counts = df['size_bin'].value_counts().sort_index()
plt.figure(figsize=(6, 6))
plt.rcParams['font.family'] = 'Calibri'

title_font = {'color': 'black', 'weight': 'bold', 'size': 16}
label_font = {'color': 'black', 'weight': 'normal', 'size': 12}

bars = plt.bar(bin_counts.index.astype(str), bin_counts.values, width=1, color='#a493f9')
for bar, value in zip(bars, bin_counts.values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(value),
              ha='center', va='bottom', fontsize=10)

plt.xlabel("Cluster Size (count of sequences)", fontdict=label_font)
plt.ylabel("Count of Clusters", fontdict=label_font)
xticks = np.arange(-0.5, len(bin_counts), 1)
xticklabels = [0, 1, 5, 10, 100, 1000, 10000]
plt.xticks(xticks, xticklabels, rotation=45)
plt.title("Distribution of Cluster Sizes(high_id)", fontdict=title_font)
plt.tight_layout()
plt.show()
os.makedirs("png", exist_ok=True)
plt.savefig(f"png/Distribution_of_Cluster_Sizes({dataset_name.split('.')[0]}_high_id).png")

## 4. Pairwise-align every cluster member to its reference sequence -> `colabel_mask_all.pkl`

Reference per cluster = the longest member. Global pairwise alignment (Biopython) gives,
for every member, the block-aligned position lists between its own structure-derived
sequence and the reference sequence.

In [ ]:
import importlib
import h5py
from tqdm import tqdm
import src.data_encoding_redefine
importlib.reload(src.data_encoding_redefine)
from src.data_encoding_redefine import aa_three_to_one, res_set, std_resnames
from Bio import Align
from Bio import SeqIO


def load_seq_from_h5(hgrp):
    M = np.array(hgrp['M'])
    qr = np.array(hgrp['qr'])
    residue_id = M[:, 1]
    residue_type = qr[:, 1]
    _, idx = np.unique(residue_id, return_index=True)
    sequence = residue_type[np.sort(idx)]
    align_b = [aa_three_to_one[res_dict[x]] for x in sequence]
    return ('').join(align_b)


aligner = Align.PairwiseAligner()
aligner.mode = 'global'

df = pd.read_csv(f"{dataset_name.split('.')[0]}_cluster_summary.csv")
df['seq_name'] = df['seq_name'].apply(ast.literal_eval)
df['seq_length'] = df['seq_length'].apply(ast.literal_eval)
df['longest'] = df.apply(lambda row: row['seq_name'][row['seq_length'].index(max(row['seq_length']))], axis=1)

results = {}
res_dict = {i: name for i, name in enumerate(std_resnames)}
res_dict[28] = 'X'
print(res_dict)

with h5py.File(dataset_path, 'r') as hf:
    for i, row in tqdm(df.iterrows(), total=len(df)):
        cluster_name = row['cluster_names'].split('/')
        clusterid = cluster_name[1] + ":" + cluster_name[2]

        ref_chain = row['longest'].split('/')[1] + "_" + row['longest'].split('/')[2].split(':')[0]
        ref_seq = all_seq_dict[f'{ref_chain}']

        seqs = row['seq_name']
        for m in range(len(seqs)):
            cur_chain = seqs[m]
            hgrp = hf[f'data/structures/{cur_chain}']
            align_b = load_seq_from_h5(hgrp)
            alignment = aligner.align(ref_seq, align_b)[0]
            aln_str1, aln_str2 = alignment.aligned
            results[cur_chain] = {
                "cluster": clusterid,
                "ref": ref_chain,
                "score": alignment.score,
                "aligned_positions_1": aln_str1.tolist(),
                "aligned_positions_2": aln_str2.tolist(),
            }

with open(f"colabel_mask_{dataset_name.split('.')[0]}.pkl", 'wb') as m:
    pickle.dump(results, m)
print(('X3/2X3W/D:0' in results))

## 5. Co-label: OR-union member labels onto the reference frame -> `all_colabel.pkl`

For each cluster: start from an all-zero label of shape `(len(ref_seq), n_classes)`, then for
every member scatter its own native label (`original[...][4]`) through the pairwise alignment
masks computed above and OR it into the reference-frame label. This is the "colabel" proper --
one boolean array per cluster covering every class any member ever showed at that reference
position.

In [ ]:
import config
importlib.reload(config)
import data_handler
importlib.reload(data_handler)
from data_handler import Dataset
from config import config_data


def positionToMask(pos_list, length):
    mask = np.zeros(length)
    for s, e in pos_list:
        mask[s:e] = 1
    return mask.astype(bool)


original = Dataset(config_data)
sel_m = np.ones(len(original.keys), dtype=bool)
sel_m &= (original.sizes[:, 1] >= config_data['min_num_res'])
sel_m &= (original.sizes[:, 0] <= config_data['max_size'])
sel_m &= select_by_interface_types(original, config_data['l_types'])
original.update_mask(sel_m)
original.set_types(config_data['l_types'], config_data['r_types'])
print(config_data['r_types'])
mapping = {key: idx for idx, key in enumerate(original.ukeys)}

with open(f"colabel_mask_{dataset_name.split('.')[0]}.pkl", 'rb') as c:
    colabel_mask = pickle.load(c)

df_cluster = pd.read_csv(f"{dataset_name.split('.')[0]}_cluster_summary.csv")
df_cluster['seq_name'] = df_cluster['seq_name'].apply(ast.literal_eval)

cluster_colabel = {}
with h5py.File(dataset_path, 'r+') as hf:
    for i, row in tqdm(df_cluster.iterrows(), total=len(df_cluster)):
        cluster = row['cluster_names']
        colabel = np.zeros((len(all_seq_dict[colabel_mask[cluster]['ref']]), len(original.t1_l)))
        for chain in row['seq_name']:
            chain_dict = colabel_mask[chain]
            hgrp = hf[f'data/structures/{chain}']
            M = np.array(hgrp['M'])
            structure_length = max(M[:, 1]) + 1
            mask1, mask2 = (positionToMask(chain_dict['aligned_positions_1'], colabel.shape[0]),
                             positionToMask(chain_dict['aligned_positions_2'], structure_length))
            long_label = np.zeros((colabel.shape[0], len(original.t1_l)))
            short_label = original[mapping[chain]][4]
            long_label[mask1] = short_label[mask2]
            colabel = colabel.astype(bool)
            long_label = long_label.astype(bool)
            colabel |= long_label
            if 'cluster' in hgrp:
                del hgrp['cluster']
            hgrp['cluster'] = cluster
        cluster_colabel[f'{cluster}'] = colabel

with open(f"{dataset_name.split('.')[0]}_colabel.pkl", 'wb') as c:
    pickle.dump(cluster_colabel, c)
print(cluster_colabel.keys())

## Appendix (optional): single-cluster sanity check + pocket sub-clustering preview

**Not part of the pipeline output** -- nothing below is saved to disk. This is the
example/visualization content salvaged from the old `colabel.ipynb`: pick one concrete
cluster, redraw its OR-union label as a heatmap to eyeball that step 5 did the right thing,
then (second half) a Tanimoto-distance + Butina-clustering pass over the same cluster's
per-member label vectors -- an early preview of the idea that was later developed into its
own pocket sub-clustering pipeline (`colabel_pocket_clustering.ipynb` / the KDE-based
`colabel_utils.KDE_Hierarchical` / `KDE_HDBSCAN`, and the standalone pocket-clustering
script that is still on the cluster and hasn't been pulled down yet). Edit `cluster = ...`
below to inspect any other cluster name from `df_cluster['cluster_names']`.

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm

iterdict = {0: 'protein', 1: 'nuclelc acid', 2: 'ion', 3: 'ligand'}
cluster = 'EX/3EXB/A:0'  # any value from df_cluster['cluster_names']

with h5py.File(dataset_path, 'r') as hf:
    row = df_cluster.loc[df_cluster['cluster_names'] == cluster].iloc[0]
    colabel = np.zeros((len(all_seq_dict[colabel_mask[cluster]['ref']]), len(original.t1_l)))
    plot_dict = {}
    for chain in row['seq_name']:
        chain_dict = colabel_mask[chain]
        hgrp = hf[f'data/structures/{chain}']
        M = np.array(hgrp['M'])
        structure_length = max(M[:, 1]) + 1
        mask1, mask2 = (positionToMask(chain_dict['aligned_positions_1'], colabel.shape[0]),
                         positionToMask(chain_dict['aligned_positions_2'], structure_length))
        long_label = np.zeros((colabel.shape[0], len(original.t1_l)))
        short_label = original[mapping[chain]][4]
        long_label[mask1] = short_label[mask2]
        plot_dict[chain] = (long_label, mask1)
        colabel = colabel.astype(bool)
        long_label = long_label.astype(bool)
        colabel |= long_label

    for m in iterdict:
        labels = []
        for seq in plot_dict:
            type_label = plot_dict[seq][0][:, m].astype(int)
            mask = plot_dict[seq][1]
            type_label[mask] += 1
            labels.append(type_label)
        labels = np.array(labels)

        colors = ["#FFFFFF", "#FEC5F6", "#56DFCF"]
        cmap = ListedColormap(colors)
        bounds = np.arange(-0.5, len(colors))
        norm = BoundaryNorm(bounds, cmap.N)

        plt.figure(figsize=(12, 3))
        im = plt.imshow(labels, cmap=cmap, norm=norm, aspect='auto', interpolation='none')
        cbar = plt.colorbar(im, ticks=range(len(colors)))
        cbar.set_label('Label value')
        plt.xlabel("Position")
        plt.ylabel("Sequence index")
        plt.title(f"Label comparison of {cluster} across sequences ({iterdict[m]} interface)")
        plt.show()

In [ ]:
from rdkit.ML.Cluster import Butina


def tanimoto_np(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return inter / union if union != 0 else 0


# reuse `cluster` / `plot_dict` from the cell above -- pocket sub-clustering preview
# on one interface type at a time
for m in iterdict:
    fps_type = [plot_dict[seq][0][:, m].astype(int) for seq in plot_dict]
    if len(fps_type) < 2:
        continue
    dists = []
    for i in range(1, len(fps_type)):
        for j in range(i):
            dists.append(1 - tanimoto_np(fps_type[i], fps_type[j]))
    sub_clusters = Butina.ClusterData(dists, len(fps_type), 0.2, isDistData=True)
    print(iterdict[m], '-> sub-clusters (Butina, cutoff 0.2):', sub_clusters)